# Severity Risk by Underlying Medical Conditions

This notebook analyzes how reported underlying medical conditions are associated with severe COVID-19 outcomes in the CDC COVID-19 Case Surveillance sample.

The goal is to compare three outcomes between cases with and without reported medical conditions:

- hospitalization
- ICU admission
- death

## Metric Definition

For each outcome, the rate is calculated as:

outcome_yes / cases with known outcome status

Each outcome uses its own denominator:

- hospitalization rate uses cases with known `hosp_yn` values
- ICU rate uses cases with known `icu_yn` values
- death rate uses cases with known `death_yn` values

Only records with `medcond_yn` values of `"Yes"` or `"No"` are included in this analysis.

## Data Preparation

To avoid bias from unknown outcome labels, this analysis filters each outcome separately.

Three filtered subsets are created:

- cases with known hospitalization status and known medical condition status
- cases with known ICU status and known medical condition status
- cases with known death status and known medical condition status

This ensures that each rate is calculated using the correct denominator for that specific outcome.

## Summary Table

The table below compares severe outcome rates for cases with and without reported underlying medical conditions.

Rates are shown as percentages.

## Grouped Comparison of Severe Outcomes

The grouped bar chart below compares hospitalization, ICU admission, and death rates by medical condition status.

This makes it easier to visually compare the severity gap between:

- cases with reported medical conditions
- cases without reported medical conditions

## Interpretation

Cases with underlying medical conditions show substantially higher rates of severe COVID outcomes across all three measures.

Compared with cases without reported medical conditions:

- hospitalization risk is much higher
- ICU admission risk is much higher
- death risk is dramatically higher

In this sample, the death rate difference is especially large, suggesting that reported underlying medical conditions are strongly associated with worse COVID-19 outcomes.

## Limitation

This project uses a 1 million row sample extracted from the CDC Case Surveillance dataset through the Socrata API.

Because the sample was retrieved through paginated API pulls rather than random sampling, the dataset should be treated as exploratory rather than fully representative of the full CDC population.

The results still provide a useful portfolio-quality comparison of severity patterns, but they should not be interpreted as official population estimates.

## Key Takeaway

Underlying medical conditions are associated with substantially higher rates of hospitalization, ICU admission, and death in this COVID-19 case surveillance sample.

This is the strongest risk-factor finding in the project and highlights the importance of comorbidities in severe COVID outcomes.

In [ ]:
import pandas as pd

df_clean = pd.read_parquet('../data/processed/cdc_clean_1M.parquet')

In [ ]:
df_known_med = (
    df_clean[
        (df_clean['hosp_yn'].isin(['Yes', 'No'])) &
        (df_clean['medcond_yn'].isin(['Yes', 'No'])) 
    ]
).copy()

total_hosp_cases = (
    df_known_med. 
    groupby('medcond_yn'). 
    size()
)

hospitalized_cases = (
    df_known_med[df_known_med['hosp_yn'] == 'Yes'].
    groupby('medcond_yn').
    size()
)

hospitalization_rate = (
    (hospitalized_cases/total_hosp_cases)*100
)

In [ ]:
df_known_icu = (
    df_clean[
        (df_clean['icu_yn'].isin(['Yes', 'No'])) &
        (df_clean['medcond_yn'].isin(['Yes', 'No'])) 
    ]
).copy()

total_icu_cases = (
    df_known_icu. 
    groupby('medcond_yn'). 
    size()
)

icu_cases = (
    df_known_icu[df_known_icu['icu_yn'] == 'Yes'].
    groupby('medcond_yn').
    size()
)

icu_rate = (
    (icu_cases/total_icu_cases)*100
)


In [ ]:
df_known_deaths = (
    df_clean[
        (df_clean['death_yn'].isin(['Yes', 'No'])) &
        (df_clean['medcond_yn'].isin(['Yes', 'No'])) 
    ]
).copy()

total_death_cases = (
    df_known_deaths. 
    groupby('medcond_yn'). 
    size()
)

death_cases = (
    df_known_deaths[df_known_deaths['death_yn'] == 'Yes'].
    groupby('medcond_yn').
    size()
)

death_rate = (
    (death_cases/total_death_cases)*100
)


In [ ]:
final_summary = pd.DataFrame({
    'hospitalization_rate': hospitalization_rate,
    'icu_rate': icu_rate,
    'death_rate': death_rate
})

final_summary = final_summary.reindex(['Yes', 'No'])
final_summary = final_summary.fillna(0)
final_summary["hospitalization_rate"] = final_summary["hospitalization_rate"].round(2)
final_summary["icu_rate"] = final_summary["icu_rate"].round(2)
final_summary["death_rate"] = final_summary["death_rate"].round(2)

In [ ]:
final_summary

In [ ]:
final_summary.to_csv("../outputs/severity_by_medical_condition.csv")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Prepare data
outcomes = ["Hospitalization", "ICU", "Death"]
yes_rates = final_summary.loc["Yes", [
    "hospitalization_rate", "icu_rate", "death_rate"
]].values
no_rates = final_summary.loc["No", [
    "hospitalization_rate", "icu_rate", "death_rate"
]].values

x = np.arange(len(outcomes))
width = 0.35

# Plot
plt.figure(figsize=(10, 6))

plt.bar(x - width/2, yes_rates, width, label="Medical Condition = Yes")
plt.bar(x + width/2, no_rates, width, label="Medical Condition = No")

plt.xticks(x, outcomes)
plt.ylabel("Rate (%)")
plt.xlabel("Outcome")
plt.title("Severe COVID-19 Outcome Rates by Underlying Medical Condition")
plt.grid(axis="y", linestyle="--", alpha=0.6)
plt.legend()
plt.tight_layout()

plt.savefig("../outputs/severity_by_medical_condition.png")
plt.show()